# OMR pipeline walkthrough

This notebook is the presentation version of the detector. It follows one photographed sheet from raw camera image to a normalized page, detected printed regions, answer grids, and bubble coordinates.

The important idea is that the production code does not guess a fixed rectangle and hope the sheet matches it. It first stabilizes the page, then reads the printed structure of the form, and only then searches for bubbles inside those recovered regions.

## 0. Imports and setup

The notebook imports the same functions used by the Streamlit app. A few internal helpers are imported for teaching purposes so we can visualize what the algorithm is doing between the public API calls.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from omr_bubble_detector import (
    detect_omr_bubbles,
    detect_sheet_layout,
    draw_detection_overlay,
    warp_sheet,
    _extract_printed_lines,
    _normalize_lighting,
)

SAMPLES = ROOT / "samples"

def bgr_to_rgb(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def show_image(image, title=None, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(bgr_to_rgb(image))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12)
    return ax

def draw_line_components(warped_bgr, horizontal_lines, vertical_lines):
    canvas = warped_bgr.copy()
    for line in horizontal_lines:
        cv2.rectangle(canvas, (line["x1"], line["y1"]), (line["x2"], line["y2"]), (0, 180, 255), 2)
    for line in vertical_lines:
        cv2.rectangle(canvas, (line["x1"], line["y1"]), (line["x2"], line["y2"]), (255, 80, 0), 2)
    return canvas

## 1. Choose a real camera sample

`cam_sample3_100.jpeg` is intentionally a tough example: it is rotated, photographed from an angle, and still contains enough structure for the robust warp path to recover the sheet. You can change `SAMPLE_NAME` to any image in `samples/`.

In [ ]:
available_samples = sorted(path.name for path in SAMPLES.glob("cam_sample*.jpeg"))
available_samples

In [ ]:
SAMPLE_NAME = "cam_sample3_100.jpeg"
image_path = SAMPLES / SAMPLE_NAME
image_bgr = cv2.imread(str(image_path))

if image_bgr is None:
    raise FileNotFoundError(image_path)

print(f"Loaded {image_path.relative_to(ROOT)} with shape {image_bgr.shape}")
show_image(image_bgr, "Original phone photo");

## 2. Perspective correction

`warp_sheet()` is the first major step. It tries the printed frame first. If that frame looks suspicious, it falls back to the paper contour and then refines the printed frame inside the cleaner paper warp.

That two-step recovery is why heavily angled photos are still usable.

In [ ]:
warped_bgr, warp_meta = warp_sheet(image_bgr)
pd.Series(warp_meta)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
show_image(image_bgr, "Original", axes[0])
show_image(warped_bgr, f"Warped sheet\nmethod: {warp_meta['warp_method']}", axes[1])
plt.tight_layout()

## 3. Printed-region discovery

After warping, the algorithm reads the form layout from the printed lines. The recovered regions are:

- header block
- Student ID cell
- Test ID cell
- answer region

This is the part that replaced the earlier fixed-height rectangles.

In [ ]:
layout = detect_sheet_layout(warped_bgr)
layout_table = pd.DataFrame(
    [
        {"region": name, "roi": layout.get(name), "source": layout.get("source", {}).get(name)}
        for name in ["header", "student_id", "test_id", "answers"]
    ]
)
layout_table

In [ ]:
horizontal_lines, vertical_lines = _extract_printed_lines(warped_bgr)
line_overlay = draw_line_components(warped_bgr, horizontal_lines, vertical_lines)

print(f"Horizontal line components: {len(horizontal_lines)}")
print(f"Vertical line components: {len(vertical_lines)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
show_image(line_overlay, "Printed line components", axes[0])
overlay = draw_detection_overlay(warped_bgr, {"layout": layout, "student_id": {"roi": layout['student_id'], "bubbles": []}, "test_id": {"roi": layout['test_id'], "bubbles": []}, "answers": {"roi": layout['answers'], "bubbles": [], "question_count": 0, "bubble_count": 0}})
show_image(overlay, "Recovered regions", axes[1])
plt.tight_layout()

## 4. Bubble grid inference

`detect_omr_bubbles()` runs the complete detector:

1. warp the sheet,
2. detect the printed layout,
3. find circular candidates inside the relevant regions,
4. infer regular grids,
5. calculate `fill_score` for each bubble.

`fill_score` is a local darkness score. It does not grade by itself; it gives the grading layer evidence about which bubble in a row is most likely marked.

In [ ]:
result = detect_omr_bubbles(image_bgr)
overlay_bgr = draw_detection_overlay(warped_bgr, result, draw_labels=True)

summary = {
    "warp_method": result["metadata"]["warp_method"],
    "layout_method": result["metadata"]["layout_method"],
    "answer_questions": result["answers"]["question_count"],
    "answer_bubbles": result["answers"]["bubble_count"],
    "student_id_bubbles": result["student_id"]["bubble_count"],
    "test_id_bubbles": result["test_id"]["bubble_count"],
    "rows_per_answer_group": result["answers"].get("rows_per_group"),
}
pd.Series(summary)

In [ ]:
show_image(overlay_bgr, "Detected bubbles and regions");

## 5. One figure for the presentation

This is the compact story slide: raw photo, normalized sheet, line evidence, recovered regions, and final bubble detections.

In [ ]:
gray = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2GRAY)
normalized = _normalize_lighting(gray)
normalized_bgr = cv2.cvtColor(normalized, cv2.COLOR_GRAY2BGR)

region_overlay = draw_detection_overlay(
    warped_bgr,
    {
        "layout": layout,
        "student_id": {"roi": layout["student_id"], "bubbles": []},
        "test_id": {"roi": layout["test_id"], "bubbles": []},
        "answers": {"roi": layout["answers"], "bubbles": [], "question_count": 0, "bubble_count": 0},
    },
)

fig, axes = plt.subplots(2, 3, figsize=(15, 12))
show_image(image_bgr, "1. Phone photo", axes[0, 0])
show_image(warped_bgr, "2. Perspective-corrected sheet", axes[0, 1])
show_image(normalized_bgr, "3. Lighting-normalized grayscale", axes[0, 2])
show_image(line_overlay, "4. Printed line evidence", axes[1, 0])
show_image(region_overlay, "5. Header + answer regions", axes[1, 1])
show_image(overlay_bgr, "6. Bubble grid detections", axes[1, 2])
plt.tight_layout()

## 6. Inspect the data returned by the detector

The detector returns normal Python dictionaries, which is useful for debugging and for the Streamlit app. The next tables are the objects that later become grading evidence.

In [ ]:
answer_df = pd.DataFrame(result["answers"]["bubbles"])
id_df = pd.DataFrame(result["student_id"]["bubbles"])
test_df = pd.DataFrame(result["test_id"]["bubbles"])

display(answer_df.head(10))
display(id_df.head(10))
display(test_df.head(10))

In [ ]:
answer_df.groupby("question")["fill_score"].agg(["min", "median", "max"]).head(12)

## 7. Functions worth mentioning in the presentation

| Function | Why it matters |
|---|---|
| `warp_sheet` | Turns a phone photo into a stable sheet coordinate system. |
| `detect_sheet_layout` | Finds header and answer regions from printed lines instead of using fixed rectangles. |
| `_extract_printed_lines` | Uses thresholding + directional morphology to isolate form lines from text and bubbles. |
| `_detect_answer_grid_with_retries` | Recovers answer rows when the top answer boundary is cropped or faint. |
| `_fill_score` | Measures local darkness inside each bubble; the grading notebook uses this as evidence. |
| `detect_omr_bubbles` | The public detection API used by the app and notebooks. |

A useful way to explain the design: each stage reduces uncertainty before the next one starts. We do not try to grade directly from the raw image.